# Day 2-01｜YOLO26 Detection 推論
> Python 籃球運動資料分析課程  
> 本單元使用已訓練 YOLO detector 對實際籃球影片執行物件偵測，觀察 class、confidence 與 bbox 輸出。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 載入 Ultralytics YOLO detector 權重。
- 對參考影片 frame 執行 detection。
- 產生一段含偵測框的 preview 影片與 JSON 結果。


## 課程流程
1. 選擇 `assets/raw/reference_videos/` 內的參考影片。
2. 對單一 frame 執行 YOLO detection。
3. 對短片段輸出 detection preview video。


In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive  # type: ignore[import-not-found]

    drive.mount("/content/drive")
except ModuleNotFoundError:
    pass

bootstrap_candidates = [
    Path("/content/drive/MyDrive/basketball_hackathon/course"),
    Path.cwd().resolve(),
    *Path.cwd().resolve().parents,
]
for candidate in bootstrap_candidates:
    if (candidate / "src" / "course_setup.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError(
        "找不到 src/course_setup.py。請先執行 init_colab.ipynb，確認課程已同步到 /content/drive/MyDrive/basketball_hackathon/course。"
    )

from src.course_setup import DEFAULT_COURSE_ROOT, bootstrap_course_notebook  # noqa: E402

COURSE_ROOT = bootstrap_course_notebook(DEFAULT_COURSE_ROOT, mount_drive=False)


In [ ]:
import pandas as pd

from src.cv_utils import save_image_rgb, show_image
from src.video_utils import display_video_in_notebook
from src.yolo_utils import (
    detector_model_path,
    rgb_from_ultralytics_plot,
    first_reference_video,
    read_video_frame,
    records_to_dicts,
    run_detector_on_image,
    write_detection_preview_video,
)

VIDEO_PATH = first_reference_video(COURSE_ROOT)
MODEL_PATH = detector_model_path(COURSE_ROOT)
FRAME_INDEX = 15

frame = read_video_frame(VIDEO_PATH, frame_index=FRAME_INDEX)
detections, result = run_detector_on_image(
    MODEL_PATH,
    frame,
    conf=0.25,
    imgsz=960,
    frame_index=FRAME_INDEX,
)
vis = rgb_from_ultralytics_plot(result)
show_image(vis, "YOLO26 detection result")

out_image = COURSE_ROOT / "assets" / "results" / "d2_01_yolo26_detection_frame.png"
save_image_rgb(out_image, vis)

print("video:", VIDEO_PATH)
print("model:", MODEL_PATH)
print("detections:", len(detections))
print("saved:", out_image)
pd.DataFrame(records_to_dicts(detections)).head()


In [ ]:
preview_path, preview_records = write_detection_preview_video(
    video_path=VIDEO_PATH,
    model_path=MODEL_PATH,
    output_path=COURSE_ROOT / "assets" / "results" / "d2_01_yolo26_detection_preview.mp4",
    max_frames=120,
    conf=0.25,
    imgsz=960,
)

print("preview video:", preview_path)
print("preview json:", preview_path.with_suffix(".json"))
print("records:", len(preview_records))
display_video_in_notebook(preview_path, loop=True)


觀察重點：`confidence` 低的框不一定錯，但可能增加後續 tracking 的誤配；球與號碼尺寸小，通常比 player 類別更容易漏偵。